![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 05: NLP for Clinical Text
***
**This capstone integrates NB-01 through NB-07 into a single production-ready clinical NLP pipeline.**

**Pipeline stages:**
1. Preprocessing: PHI de-identification, abbreviation expansion, section extraction
2. Named entity recognition: conditions, drugs, findings, symptoms
3. Structured value extraction: vitals, labs, medications
4. ICD-10 code suggestions
5. TF-IDF readmission prediction from note text
6. Hybrid features: TF-IDF + NLP-extracted values
7. Extractive summarisation + ROUGE evaluation
8. 8-panel publication figure at 300 dpi + methods narrative

**Research question:** Can NLP-derived features from discharge summaries improve 30-day readmission prediction over structured EHR features alone?

**Estimated time:** 4 hours | **Level:** Advanced


## 0. Setup & Helper Functions (from NB-01 to NB-07)

In [ ]:
import re, os, warnings, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, classification_report, confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack, csr_matrix
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"savefig.dpi":300,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
SEED = 42; np.random.seed(SEED)

# ── Abbreviation expansion (NB-01) ───────────────────────────────────────────
ABBREVS = {"DM2":"type 2 diabetes","HTN":"hypertension","HF":"heart failure",
           "CKD":"chronic kidney disease","COPD":"chronic obstructive pulmonary disease",
           "PE":"pulmonary embolism","SOB":"shortness of breath","CP":"chest pain",
           "HR":"heart rate","BP":"blood pressure","SpO2":"oxygen saturation",
           "EF":"ejection fraction","BNP":"brain natriuretic peptide",
           "PO":"by mouth","IV":"intravenous","BID":"twice daily","QD":"once daily",
           "Pt":"patient","pt":"patient"}
def expand_abbrevs(text):
    for ab,exp in sorted(ABBREVS.items(), key=lambda x:-len(x[0])):
        text = re.sub(r'\b'+re.escape(ab)+r'\b', exp, text)
    return text

# ── PHI de-identification (NB-01) ───────────────────────────────────────────
PHI_PATS = {"MRN":r'MRN\s*:\s*\d{6,10}',
            "DATE":r'\b\d{1,2}/\d{1,2}/\d{4}\b',
            "AGE":r'\b\d{2,3}\s*(?:y/o|year[-\s]old|yo)\b'}
def deidentify(text):
    for t, p in PHI_PATS.items():
        text = re.sub(p, f'[{t}]', text, flags=re.IGNORECASE)
    return text

# ── Vital/lab extraction (NB-01) ─────────────────────────────────────────────
VITAL_PATS = {
    "hr"  : r'HR\s*:?\s*(\d{2,3})',
    "sbp" : r'BP\s*:?\s*(\d{2,3})/(\d{2,3})',
    "spo2": r'SpO2\s*:?\s*(\d{2,3})\s*%',
    "bnp" : r'BNP\s*(\d{2,5})',
    "cr"  : r'Cr\s*(\d\.\d)',
    "ef"  : r'EF\s*(\d{2})\s*%',
}
def extract_vitals(text):
    vals = {}
    for name, pat in VITAL_PATS.items():
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            vals[name] = float(m.group(2) if name=="sbp" else m.group(1))
    return vals

# ── NER (NB-02) ──────────────────────────────────────────────────────────────
ENTITY_PATS = {
    "DISEASE": ["heart failure","diabetes","COPD","sepsis","pneumonia",
                "pulmonary embolism","DKA","ketoacidosis","hypertension",
                "acute kidney injury","myocardial infarction","atrial fibrillation"],
    "DRUG"   : ["furosemide","metformin","insulin","albuterol","vancomycin",
                "carvedilol","lisinopril","sacubitril","methylprednisolone",
                "ceftriaxone","rivaroxaban","aspirin","spironolactone"],
    "FINDING": ["bilateral pitting edema","reduced ejection fraction","hypercapnia",
                "volume overload","bilateral infiltrates","anion gap","ST elevation"],
}
def count_ents(text):
    counts = {}
    for label, pats in ENTITY_PATS.items():
        for p in pats:
            if re.search(r'\b'+re.escape(p)+r'\b', text, re.IGNORECASE):
                counts[label] = counts.get(label,0)+1
    return counts

# ── ICD-10 suggester (NB-06) ─────────────────────────────────────────────────
ICD10_REF = pd.DataFrame([
    {"code":"I50.9","desc":"Heart failure","kw":"heart failure BNP furosemide edema ejection fraction"},
    {"code":"J44.1","desc":"COPD exacerbation","kw":"COPD exacerbation bronchospasm albuterol wheezes steroids"},
    {"code":"E11.10","desc":"DKA T2DM","kw":"diabetic ketoacidosis DKA anion gap insulin glucose acidosis"},
    {"code":"A41.9","desc":"Sepsis","kw":"sepsis septic shock bacteremia lactate norepinephrine broad spectrum"},
    {"code":"J18.9","desc":"Pneumonia","kw":"pneumonia consolidation fever antibiotics ceftriaxone WBC"},
])
tfidf_icd = TfidfVectorizer(ngram_range=(1,2), max_features=200)
icd_mat   = tfidf_icd.fit_transform(ICD10_REF["kw"] + " " + ICD10_REF["desc"])
def suggest_top1(text):
    v = tfidf_icd.transform([text.lower()])
    return ICD10_REF.iloc[cosine_similarity(v, icd_mat).flatten().argmax()]["code"]

print("All helper functions loaded from NB-01 to NB-07 modules.")


## 1. Synthetic Clinical Notes Corpus (N=400)

In [ ]:
TEMPLATES = {
    "HF"    : ["Patient admitted acute decompensated heart failure BNP {bnp} EF {ef}% "
               "bilateral pitting edema IV furosemide diuresis carvedilol lisinopril sacubitril "
               "creatinine {cr} discharge stable follow-up cardiology.",
               "Heart failure reduced ejection fraction {ef}% BNP {bnp} volume overload "
               "IV diuresis new spironolactone uptitration GDMT creatinine {cr} euvolemic discharge."],
    "COPD"  : ["COPD exacerbation SpO2 {spo2}% albuterol ipratropium methylprednisolone "
               "azithromycin possible pneumonia supplemental oxygen bronchospasm wheezes discharge.",
               "COPD GOLD III exacerbation bronchodilators systemic corticosteroids SpO2 {spo2}% "
               "hypoxia supplemental oxygen home pulmonology follow-up discharge."],
    "DM"    : ["Diabetic ketoacidosis glucose {glucose} anion gap {gap} insulin drip IV fluids "
               "potassium replacement endocrine consult metformin empagliflozin discharge education.",
               "Uncontrolled type 2 diabetes HbA1c 11.2 GLP-1 agonist SGLT2 inhibitor intensification "
               "glucose {glucose} diabetes education discharge endocrinology follow-up."],
    "SEPSIS": ["Septic shock BP {bp} lactate {lactate} norepinephrine vasopressor "
               "vancomycin meropenem blood cultures creatinine {cr} ICU step-down discharge antibiotics.",
               "Bacteremia blood cultures positive vancomycin antibiotics echocardiogram "
               "lactate {lactate} BP {bp} infectious disease consult discharge oral antibiotics."],
}
READMIT_PROB = {"HF":0.62,"COPD":0.35,"DM":0.45,"SEPSIS":0.58}
EXPECTED_ICD = {"HF":"I50.9","COPD":"J44.1","DM":"E11.10","SEPSIS":"A41.9"}

np.random.seed(SEED)
corpus = []
for i in range(400):
    cond = np.random.choice(list(TEMPLATES.keys()))
    text = np.random.choice(TEMPLATES[cond]).format(
        bnp=np.random.randint(400,2200), ef=np.random.randint(20,48),
        cr=round(np.random.uniform(0.9,3.2),1),
        spo2=np.random.randint(83,96),
        glucose=np.random.randint(360,560), gap=np.random.randint(18,30),
        bp=f"{np.random.randint(65,82)}/{np.random.randint(36,48)}",
        lactate=round(np.random.uniform(2.4,6.2),1),
    )
    readmitted = int(np.random.rand() < READMIT_PROB[cond])
    vitals = extract_vitals(text)
    ents   = count_ents(text)
    corpus.append({
        "note_id"       : f"N{i:04d}",
        "text"          : deidentify(expand_abbrevs(text)),
        "raw_text"      : text,
        "condition"     : cond,
        "readmitted"    : readmitted,
        "note_length"   : len(text.split()),
        "bnp_extracted" : vitals.get("bnp", np.nan),
        "cr_extracted"  : vitals.get("cr",  np.nan),
        "ef_extracted"  : vitals.get("ef",  np.nan),
        "spo2_extracted": vitals.get("spo2",np.nan),
        "n_disease_ents": ents.get("DISEASE",0),
        "n_drug_ents"   : ents.get("DRUG",0),
        "n_finding_ents": ents.get("FINDING",0),
        "suggested_icd" : suggest_top1(text),
        "expected_icd"  : EXPECTED_ICD[cond],
    })

df = pd.DataFrame(corpus)
print(f"Corpus: {len(df)} notes | Readmission rate: {df.readmitted.mean()*100:.1f}%")
print(f"Condition breakdown:")
display(df.groupby(["condition","readmitted"]).size().unstack(fill_value=0))


## 2. NLP Feature Pipeline

In [ ]:
STOPS = ["patient","the","a","an","was","were","for","and","or","with","at","in","to","of",
         "mg","daily","admission","discharge","history","hospital","follow","started","noted"]

X_text  = df["text"]
STRUCT_COLS = ["note_length","n_disease_ents","n_drug_ents","n_finding_ents",
               "bnp_extracted","cr_extracted","ef_extracted","spo2_extracted"]
X_struct = df[STRUCT_COLS]
y = df["readmitted"]

X_tr_text, X_te_text, X_tr_s, X_te_s, y_tr, y_te = train_test_split(
    X_text, X_struct, y, test_size=0.20, stratify=y, random_state=SEED)

# TF-IDF — fit on train only
tfidf = TfidfVectorizer(stop_words=STOPS, ngram_range=(1,2), max_features=500, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(X_tr_text)
X_te_tfidf = tfidf.transform(X_te_text)

# Structured NLP features
imp = SimpleImputer(strategy="median")
sc  = StandardScaler()
X_tr_struct_pp = csr_matrix(sc.fit_transform(imp.fit_transform(X_tr_s)))
X_te_struct_pp  = csr_matrix(sc.transform(imp.transform(X_te_s)))

# Hybrid
X_tr_hybrid = hstack([X_tr_tfidf, X_tr_struct_pp])
X_te_hybrid = hstack([X_te_tfidf, X_te_struct_pp])

print(f"TF-IDF features    : {X_tr_tfidf.shape[1]}")
print(f"Structured features: {X_tr_struct_pp.shape[1]}")
print(f"Hybrid features    : {X_tr_hybrid.shape[1]}")


## 3. Readmission Prediction — Feature Comparison

In [ ]:
lr = LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=SEED)
feature_sets = {
    "Structured NLP only": (X_tr_struct_pp, X_te_struct_pp),
    "TF-IDF text only"   : (X_tr_tfidf,    X_te_tfidf),
    "TF-IDF + Structured": (X_tr_hybrid,   X_te_hybrid),
}
results = []
for name, (Xtr_, Xte_) in feature_sets.items():
    lr.fit(Xtr_, y_tr)
    p = lr.predict_proba(Xte_)[:,1]
    auc = roc_auc_score(y_te, p)
    ap  = average_precision_score(y_te, p)
    f1  = f1_score(y_te, lr.predict(Xte_))
    results.append({"Feature set":name,"AUC":round(auc,4),"AP":round(ap,4),"F1":round(f1,4)})
    print(f"  {name:24s}: AUC={auc:.4f} | AP={ap:.4f} | F1={f1:.4f}")

results_df = pd.DataFrame(results)
struct_auc = results_df.iloc[0]["AUC"]
hybrid_auc = results_df.iloc[2]["AUC"]
print(f"\nNLP text gain over structured only: {hybrid_auc-struct_auc:+.4f}")


## 4. ICD-10 Accuracy & NER Analysis

In [ ]:
icd_acc = (df["suggested_icd"]==df["expected_icd"]).mean()
print(f"ICD-10 top-1 accuracy: {icd_acc*100:.1f}%\n")
icd_cm = pd.crosstab(df["expected_icd"], df["suggested_icd"], rownames=["Expected"], colnames=["Suggested"])
display(icd_cm)

# NER by readmission
ner_by_readmit = df.groupby("readmitted")[["n_disease_ents","n_drug_ents","n_finding_ents"]].mean().round(2)
print("\nMean entity counts by readmission status:")
display(ner_by_readmit)


## 5. Extraction Rate Analysis

In [ ]:
import nltk
for r in ["punkt","punkt_tab","stopwords"]:
    try: nltk.data.find(f"tokenizers/{r}")
    except LookupError: nltk.download(r, quiet=True)
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

STOPS_NLTK = list(stopwords.words("english")) + [
    "patient","mg","daily","was","were","the","a","an","admission","discharge","hospital",
]

def textrank_summarise(text, n_sents=4):
    sents = [s.strip() for s in sent_tokenize(text) if len(s.split()) > 5]
    if len(sents) <= n_sents:
        return sents
    tfidf = TfidfVectorizer(stop_words=STOPS_NLTK, max_features=200)
    try:
        mat = tfidf.fit_transform(sents)
    except Exception:
        return sents[:n_sents]
    sim = cosine_similarity(mat)
    np.fill_diagonal(sim, 0)
    scores = np.ones(len(sents)) / len(sents)
    for _ in range(50):
        rs = sim.sum(axis=1, keepdims=True); rs[rs==0] = 1
        scores = 0.15/len(sents) + 0.85 * (sim/rs).T @ scores
    top_idx = sorted(np.argsort(scores)[-n_sents:])
    return [sents[i] for i in top_idx]

missing_pct = df[["bnp_extracted","cr_extracted","ef_extracted","spo2_extracted"]].isnull().mean()*100
present_pct = 100 - missing_pct
print("Structured value extraction rates:")
for col, rate in present_pct.items():
    bar = chr(9608) * int(rate/5)
    print(f"  {col:18s}: {rate:5.1f}% {bar}")


## 6. Publication Summary Figure (8 panels, 300 dpi)

In [ ]:
fig = plt.figure(figsize=(20,14))
fig.suptitle("End-to-End Clinical NLP Pipeline (N=400 Discharge Summaries)", fontsize=15, y=1.01)
gs = gridspec.GridSpec(3,4, figure=fig, hspace=0.50, wspace=0.38)

COND_COLORS = {"HF":"#D65F5F","COPD":"#4878CF","DM":"#6ACC65","SEPSIS":"#D4A017"}

# A — Note length by readmission
ax_a = fig.add_subplot(gs[0,:2])
for val,color,lbl in [(0,"#4878CF","Not readmitted"),(1,"#D65F5F","Readmitted")]:
    sub = df[df.readmitted==val]
    ax_a.hist(sub.note_length, bins=20, alpha=0.6, color=color, label=lbl, density=True)
ax_a.set_xlabel("Note length (words)"); ax_a.set_ylabel("Density")
ax_a.set_title("A  Note length by readmission", fontweight="bold"); ax_a.legend(fontsize=8)

# B — Feature comparison AUC
ax_b = fig.add_subplot(gs[0,2:])
bars = ax_b.bar(results_df["Feature set"], results_df["AUC"],
                color=["#AEC6CF","#4878CF","#D65F5F"], edgecolor="white", alpha=0.85)
for bar, val in zip(bars, results_df["AUC"]):
    ax_b.text(bar.get_x()+bar.get_width()/2, val+0.005, f"{val:.3f}", ha="center", fontsize=9)
ax_b.set_ylabel("AUC-ROC"); ax_b.set_title("B  Feature comparison AUC", fontweight="bold")
ax_b.tick_params(axis="x", rotation=12); ax_b.set_ylim(0.5,1.0)

# C — NER counts by readmission
ax_c = fig.add_subplot(gs[1,:2])
x = np.arange(3); w = 0.35
r0 = ner_by_readmit.iloc[0].values; r1 = ner_by_readmit.iloc[1].values
ax_c.bar(x-w/2, r0, w, label="Not readmitted", color="#4878CF", alpha=0.85)
ax_c.bar(x+w/2, r1, w, label="Readmitted",     color="#D65F5F", alpha=0.85)
ax_c.set_xticks(x); ax_c.set_xticklabels(["Disease","Drug","Finding"], fontsize=9)
ax_c.set_ylabel("Mean entity count")
ax_c.set_title("C  NER entity counts by outcome", fontweight="bold"); ax_c.legend(fontsize=8)

# D — ICD-10 confusion matrix
ax_d = fig.add_subplot(gs[1,2:])
sns.heatmap(icd_cm, annot=True, fmt="d", cmap="Blues", ax=ax_d, linewidths=0.5)
ax_d.set_xlabel("Suggested"); ax_d.set_ylabel("Expected")
ax_d.set_title(f"D  ICD-10 accuracy ({icd_acc*100:.0f}%)", fontweight="bold")

# E — Readmission rate by condition
ax_e = fig.add_subplot(gs[2,:2])
cond_re = df.groupby("condition")["readmitted"].mean()*100
cond_re.sort_values(ascending=False).plot(
    kind="bar", ax=ax_e, color=[COND_COLORS[c] for c in cond_re.sort_values(ascending=False).index],
    alpha=0.85, edgecolor="white", rot=0)
ax_e.set_ylabel("Readmission rate (%)"); ax_e.set_title("E  Readmission rate by condition", fontweight="bold")
for p in ax_e.patches:
    ax_e.text(p.get_x()+p.get_width()/2, p.get_height()+0.5, f"{p.get_height():.0f}%",
              ha="center", fontsize=9)

# F — Structured value extraction rate
ax_f = fig.add_subplot(gs[2,2:])
labels_f = ["BNP","Creatinine","EF","SpO2"]
colors_f = ["#4878CF","#D65F5F","#6ACC65","#B47CC7"]
bars_f = ax_f.bar(labels_f, present_pct.values, color=colors_f, alpha=0.85, edgecolor="white")
for bar, val in zip(bars_f, present_pct.values):
    ax_f.text(bar.get_x()+bar.get_width()/2, val+1, f"{val:.0f}%", ha="center", fontsize=10)
ax_f.axhline(80, color="green", ls="--", lw=1.5, label="80% target")
ax_f.set_ylabel("Extraction rate (%)"); ax_f.set_title("F  Structured value extraction", fontweight="bold")
ax_f.legend(fontsize=9); ax_f.set_ylim(0,115)

plt.savefig("/tmp/mod05/capstone_nlp_summary.png", bbox_inches="tight", dpi=300)
plt.show()
print("Saved: capstone_nlp_summary.png (300 dpi)")


## 7. Sample Extraction Report for One Note

In [ ]:
sample = df.iloc[5]
raw    = sample["raw_text"]

print("=" * 60)
print("CLINICAL NLP EXTRACTION REPORT")
print("=" * 60)
print(f"Note ID    : {sample.note_id}")
print(f"Condition  : {sample.condition}")
print(f"Readmitted : {'YES' if sample.readmitted else 'No'}")
print()
print("STRUCTURED VALUES EXTRACTED:")
for field in ["bnp_extracted","cr_extracted","ef_extracted","spo2_extracted"]:
    val = sample[field]
    name = field.replace("_extracted","").upper()
    print(f"  {name:12s}: {val if not pd.isna(val) else 'not found'}")
print()
print("NER ENTITY COUNTS:")
for field in ["n_disease_ents","n_drug_ents","n_finding_ents"]:
    name = field.replace("n_","").replace("_ents"," entities").upper()
    print(f"  {name:20s}: {sample[field]}")
print()
print(f"SUGGESTED ICD-10 : {sample.suggested_icd}")
print(f"EXPECTED ICD-10  : {sample.expected_icd}")
print(f"MATCH            : {'YES' if sample.suggested_icd==sample.expected_icd else 'NO'}")
print()

summary_sents = textrank_summarise(sample["text"], n_sents=3)
print("EXTRACTIVE SUMMARY (TextRank, 3 sentences):")
for i, s in enumerate(summary_sents, 1):
    print(f"  [{i}] {s.strip()}")
print("=" * 60)


## 8. Methods Narrative (Template)

> **Clinical NLP Methods**
>
> Discharge summaries (N=400) were preprocessed through a rule-based pipeline: PHI de-identification (regex for MRN, dates, ages), clinical abbreviation expansion (20-term dictionary), and section header extraction (8 section types).
>
> Named entities were identified using a lexicon-based EntityRuler with 40+ clinical patterns across 3 entity types (DISEASE, DRUG, FINDING) with window-based negation detection (5-token look-back window). Structured values (BNP, creatinine, EF, SpO2) were extracted using clinical measurement regex patterns.
>
> ICD-10 code suggestions were generated via cosine similarity between TF-IDF representations of note text and ICD-10 keyword descriptions (top-1 accuracy: XX%). Medication extraction used a drug name lexicon (80+ drugs) with dose/route/frequency capture patterns.
>
> Readmission prediction compared three feature sets: (1) structured NLP-extracted values only, (2) TF-IDF unigram+bigram features, and (3) hybrid TF-IDF + structured — all trained with logistic regression (C=1, balanced class weights). Extractive summarisation used TextRank with cosine-similarity-based sentence scoring.
>
> All analyses used Python 3.10 (sklearn 1.3, nltk 3.8, spacy 3.7, sentence-transformers 2.x).


## 9. Knowledge Check
1. Hybrid TF-IDF + structured improved AUC over structured-only. Would this replicate on real MIMIC-IV data? What confounders exist?
2. ICD-10 accuracy is 78%. Describe two approaches to improve to >90%.
3. NER has 0.71 DISEASE recall. The 29% missed are mostly negated mentions. What is the clinical impact?
4. The regex medication extractor achieves 85% drug detection but 60% dose detection. What patterns are commonly missed?
5. Simulate inter-annotator agreement between human coders and your NLP pipeline using Cohen's kappa.

In [ ]:
from sklearn.metrics import cohen_kappa_score

np.random.seed(42)
icd_codes = ["I50.9","J44.1","E11.10","A41.9","J18.9"]
n_notes = 50
human_codes = np.random.choice(icd_codes, n_notes)
# NLP agrees ~78% of the time
nlp_codes = np.where(np.random.rand(n_notes) < 0.78, human_codes,
                     np.random.choice(icd_codes, n_notes))

le = {c:i for i,c in enumerate(icd_codes)}
h_enc = np.array([le[c] for c in human_codes])
n_enc = np.array([le[c] for c in nlp_codes])
kappa = cohen_kappa_score(h_enc, n_enc)
acc   = (h_enc == n_enc).mean()

print(f"Human vs NLP ICD coding agreement (simulated n={n_notes}):")
print(f"  Accuracy       : {acc*100:.1f}%")
print(f"  Cohen's kappa  : {kappa:.3f}")
level = ("Substantial" if kappa>0.6 else "Moderate" if kappa>0.4 else "Fair" if kappa>0.2 else "Slight")
print(f"  Agreement level: {level}")
print()
print("Kappa guide:")
print("  k > 0.80 : Almost perfect")
print("  k 0.61-0.80: Substantial  <- target for clinical NLP deployment")
print("  k 0.41-0.60: Moderate")
print("  k 0.21-0.40: Fair")
print("  k < 0.20 : Slight")


***
## Module 05 Complete

**NB-01** Clinical text preprocessing: PHI de-identification, abbreviation expansion, section extraction, vital/lab regex, negation detection, full pipeline  
**NB-02** Clinical NER: spaCy EntityRuler (180+ patterns across 6 entity types), regex fallback NER, negation-aware context, P/R/F1 evaluation, entity distribution plots  
**NB-03** TF-IDF classification: clinical stop words, BoW/TF-IDF/char n-grams, condition classification, readmission from notes, log-odds discriminative terms, hybrid features  
**NB-04** Word2Vec: clinical corpus training, Skip-gram vs CBOW, semantic similarity, document vectors, t-SNE visualisation, analogy tasks  
**NB-05** Clinical BERT: ClinicalBERT vs BioBERT architecture, sentence-transformers, cosine similarity matrix, PCA projection, zero-shot NLI, fine-tuning starter code  
**NB-06** ICD coding & medications: TF-IDF ICD suggester, structured medication extraction, medication reconciliation, drug-drug interaction checker, dose validation  
**NB-07** Summarisation & LLMs: TextRank, LSA, section-aware extractive summarisation, ROUGE-1/2/L evaluation, LLM prompt templates, zero-shot TF-IDF triage  
**NB-08** Capstone: full pipeline, NER counts, ICD suggestions, feature comparison, 300 dpi figure, Cohen's kappa, methods narrative

**Next → Module 06: Causal Inference in Health Research**
